In [1]:
import rasterio
import numpy as np
import pandas as pd
from rasterio.windows import Window
from rasterio.transform import rowcol
from pyproj import Transformer, CRS
from shapely.geometry import Polygon
import pickle
import os

In [2]:
slope_path = '../00_raw_data/slope_elevation_maps/LDSM_80S_40MPP_ADJ.tiff'
elev_path  = '../00_raw_data/slope_elevation_maps/LDEM_80S_40MPP_ADJ.tiff'

for label, path in [('Slope', slope_path), ('Elevation', elev_path)]:
    with rasterio.open(path) as src:
        print(f"{label}")
        print(f"  CRS:        {src.crs}")
        print(f"  Resolution: {src.res} m/px")
        print(f"  Dimensions: {src.width} x {src.height} px")
        print(f"  Bounds:     {src.bounds}")
        print(f"  NoData:     {src.nodata}")
        print()

Slope
  CRS:        PROJCS["Moon (2015) - Sphere / Ocentric / South Polar",GEOGCS["Moon (2015) - Sphere / Ocentric",DATUM["Moon (2015) - Sphere",SPHEROID["Moon (2015) - Sphere",1737400,0,AUTHORITY["IAU","30100"]],AUTHORITY["IAU","30100"]],PRIMEM["Reference Meridian",0,AUTHORITY["IAU","30100"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["IAU","30100"]],PROJECTION["Polar_Stereographic"],PARAMETER["latitude_of_origin",-90],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",NORTH],AXIS["Northing",NORTH],AUTHORITY["IAU","30135"]]
  Resolution: (40.0, 40.0) m/px
  Dimensions: 15200 x 15200 px
  Bounds:     BoundingBox(left=-304000.0, bottom=-304000.0, right=304000.0, top=304000.0)
  NoData:     nan

Elevation
  CRS:        PROJCS["Moon (2015) - Sphere / Ocentric / South Polar",GEOGCS["Moon (2015) - Sphere / Ocentric",DATUM["Moon (2015) - Sphere",SPHEROID["Moon (2015) - Sp

In [3]:
# 9 candidate landing regions -- coordinates from LROC QuickMap
# area and perimeter from LROC QuickMap feature inspector
# ref: NASA (2024) update on Artemis III/IV candidate landing regions

raw_sites = {
    'Nobile Rim 2': {
        'corners':     [(-84.04797, 54.46178), (-84.40465, 59.96161),
                        (-83.82638, 63.02425), (-83.50158, 57.84020)],
        'area_km2':    397.770, 'perimeter_km': 79.777
    },
    'Mons Mouton': {
        'corners':     [(-85.32705, 27.28799), (-85.78834, 30.57923),
                        (-85.49737, 36.36658), (-85.06330, 32.73523)],
        'area_km2':    255.183, 'perimeter_km': 63.898
    },
    'Malapert Massif': {
        'corners':     [(-85.63394, 355.23636), (-86.32279, 354.34052),
                        (-86.32589,   5.14788), (-85.63656,   4.33231)],
        'area_km2':    439.918, 'perimeter_km': 83.896
    },
    'de Gerlache Rim 2': {
        'corners':     [(-87.78529, 299.42746), (-88.02393, 282.52881),
                        (-88.65999, 288.65789), (-88.32789, 310.60129)],
        'area_km2':    399.802, 'perimeter_km': 79.980
    },
    'Mons Mouton Plateau': {
        'corners':     [(-83.95020,  15.49417), (-85.75564,  22.39284),
                        (-84.28934,  46.61204), (-82.84671,  35.44246)],
        'area_km2':    4429.268, 'perimeter_km': 268.888
    },
    'Slater Plain': {
        'corners':     [(-87.62216, 123.69062), (-87.20229, 134.99761),
                        (-86.70324, 126.86961), (-87.05106, 116.56720)],
        'area_km2':    399.302, 'perimeter_km': 79.930
    },
    'Peak near Cabeus B': {
        'corners':     [(-83.20950, 292.83328), (-83.43644, 287.52647),
                        (-84.06005, 289.44052), (-83.81001, 295.20011)],
        'area_km2':    397.335, 'perimeter_km': 79.733
    },
    'Nobile Rim 1': {
        'corners':     [(-85.35140,  31.64367), (-85.89749,  36.47997),
                        (-85.47479,  43.20664), (-84.97460,  38.05472)],
        'area_km2':    398.725, 'perimeter_km': 79.872
    },
    'Haworth': {
        'corners':     [(-86.11240, 333.34804), (-86.95784, 325.01815),
                        (-87.39353, 343.00918), (-86.44277, 347.63822)],
        'area_km2':    886.589, 'perimeter_km': 119.102
    },
}

def normalize_lon(lon):
    return lon - 360 if lon > 180 else lon

def region_half_window_px(area_km2, perimeter_km, px_size_m):
    half_perim = perimeter_km / 2
    disc       = half_perim**2 - 4 * area_km2
    long_side  = (half_perim + np.sqrt(max(disc, 0))) / 2 if disc >= 0 \
                 else np.sqrt(area_km2)
    return int((long_side * 1000 / 2) / px_size_m) + 1

px_size_slope = 40

records       = []
site_polygons = {}

for site, meta in raw_sites.items():
    lats = [c[0] for c in meta['corners']]
    lons = [normalize_lon(c[1]) for c in meta['corners']]
    hw   = region_half_window_px(meta['area_km2'],
                                  meta['perimeter_km'], px_size_slope)
    records.append({
        'Site':         site,
        'Lat':          round(np.mean(lats), 5),
        'Lon':          round(np.mean(lons), 5),
        'Area_km2':     meta['area_km2'],
        'Perimeter_km': meta['perimeter_km'],
        'Search_hw_px': hw,
    })
    site_polygons[site] = list(zip(lats, lons))

    print(f"{site:25s}  area: {meta['area_km2']:7.1f} km²  "
          f"half-window: {hw}px ({hw*px_size_slope/1000:.1f}km)")

sites_df = pd.DataFrame(records)

Nobile Rim 2               area:   397.8 km²  half-window: 251px (10.0km)
Mons Mouton                area:   255.2 km²  half-window: 201px (8.0km)
Malapert Massif            area:   439.9 km²  half-window: 263px (10.5km)
de Gerlache Rim 2          area:   399.8 km²  half-window: 250px (10.0km)
Mons Mouton Plateau        area:  4429.3 km²  half-window: 959px (38.4km)
Slater Plain               area:   399.3 km²  half-window: 250px (10.0km)
Peak near Cabeus B         area:   397.3 km²  half-window: 250px (10.0km)
Nobile Rim 1               area:   398.7 km²  half-window: 250px (10.0km)
Haworth                    area:   886.6 km²  half-window: 373px (14.9km)


In [4]:
moon_gcs = CRS.from_proj4('+proj=longlat +R=1737400 +no_defs')

moon_spole = CRS.from_proj4(
    '+proj=stere +lat_0=-90 +lon_0=0 +k=1 '
    '+x_0=0 +y_0=0 +R=1737400 +units=m +no_defs'
)

transformer = Transformer.from_crs(moon_gcs, moon_spole, always_xy=True)

sites_df['X_stereo'], sites_df['Y_stereo'] = transformer.transform(
    sites_df['Lon'].values,
    sites_df['Lat'].values
)

site_polygons_stereo = {}
for site, corners_latlon in site_polygons.items():
    lats = [c[0] for c in corners_latlon]
    lons = [c[1] for c in corners_latlon]
    xs, ys = transformer.transform(lons, lats)
    site_polygons_stereo[site] = Polygon(list(zip(xs, ys)))

    computed_area = site_polygons_stereo[site].area / 1_000_000
    documented_area = raw_sites[site]['area_km2']
    print(f"{site:25s}  polygon area: {computed_area:.1f} km²  "
          f"(documented: {documented_area:.1f} km²)")

sites_df[['Site', 'Lat', 'Lon', 'X_stereo', 'Y_stereo', 'Area_km2']]

Nobile Rim 2               polygon area: 400.0 km²  (documented: 397.8 km²)
Mons Mouton                polygon area: 256.0 km²  (documented: 255.2 km²)
Malapert Massif            polygon area: 441.0 km²  (documented: 439.9 km²)
de Gerlache Rim 2          polygon area: 400.0 km²  (documented: 399.8 km²)
Mons Mouton Plateau        polygon area: 4451.8 km²  (documented: 4429.3 km²)
Slater Plain               polygon area: 399.8 km²  (documented: 399.3 km²)
Peak near Cabeus B         polygon area: 399.8 km²  (documented: 397.3 km²)
Nobile Rim 1               polygon area: 400.0 km²  (documented: 398.7 km²)
Haworth                    polygon area: 888.0 km²  (documented: 886.6 km²)


,Site,Lat,Lon,X_stereo,Y_stereo,Area_km2
0,Nobile Rim 2,-83.94515,58.82196,157230.527667,95139.846049,397.770
1,Mons Mouton,-85.41902,31.74226,73119.722381,118195.834537,255.183
2,Malapert Massif,-85.97980,-0.23573,-501.757729,121954.940065,439.918
3,de Gerlache Rim 2,-88.19928,-64.69614,-49368.891737,23340.637476,399.802
4,Mons Mouton Plateau,-84.21047,29.98538,87814.906877,152189.549919,4429.268
5,Slater Plain,-87.14469,125.53126,70475.354253,-50327.618069,399.302
6,Peak near Cabeus B,-83.62900,-68.74990,-180240.289696,70091.977438,397.335
7,Nobile Rim 1,-85.42457,37.34625,84210.082350,110356.622123,398.725
8,Haworth,-86.72664,-22.74660,-38389.647441,91564.153720,886.589


In [5]:
# extract slope and elevation for each site's search area, and save the
# window-corrected transform so that pixel coordinates in the cropped
# array correctly map to real-world stereographic meters

ez_radius_m  = 2000
ez_radius_px = ez_radius_m // px_size_slope

os.makedirs('../01_cleaned_data/site_grids', exist_ok=True)

grid_meta = []

with rasterio.open(slope_path) as slope_src, \
     rasterio.open(elev_path)  as elev_src:

    for _, row in sites_df.iterrows():
        search_ws = int(row['Search_hw_px'])
        buf       = ez_radius_px

        py, px_c = rowcol(slope_src.transform, row['X_stereo'], row['Y_stereo'])

        r0 = max(0, py   - search_ws - buf)
        r1 = min(slope_src.height, py + search_ws + buf)
        c0 = max(0, px_c - search_ws - buf)
        c1 = min(slope_src.width,  px_c + search_ws + buf)

        win = Window(c0, r0, c1-c0, r1-r0)
        slope_arr = slope_src.read(1, window=win).astype(float)
        elev_arr  = elev_src.read(1,  window=win).astype(float)

        if slope_src.nodata is not None: slope_arr[slope_arr == slope_src.nodata] = np.nan
        if elev_src.nodata  is not None: elev_arr[elev_arr   == elev_src.nodata]  = np.nan

        # window-corrected transform -- this maps pixel (row, col) in the
        # CROPPED array to the correct real-world coordinate
        window_transform = rasterio.windows.transform(win, slope_src.transform)

        site_key  = row['Site'].replace(' ', '_').replace('/', '_')
        grid_file = f'../01_cleaned_data/site_grids/{site_key}_grids.pkl'

        with open(grid_file, 'wb') as f:
            pickle.dump({
                'slope': slope_arr,
                'elev':  elev_arr,
                'r0': r0, 'c0': c0,
                'transform': window_transform,
                'search_ws': search_ws,
                'px_size_slope': px_size_slope,
            }, f)

        grid_meta.append({'Site': row['Site'], 'site_key': site_key})
        print(f"{row['Site']:25s}  grid shape: {slope_arr.shape}  "
              f"transform origin: ({window_transform.c:.0f}, {window_transform.f:.0f})")

sites_df = sites_df.merge(pd.DataFrame(grid_meta), on='Site')
sites_df.to_csv('../01_cleaned_data/sites_df.csv', index=False)

with open('../01_cleaned_data/site_grids/site_polygons.pkl', 'wb') as f:
    pickle.dump(site_polygons_stereo, f)

print("\nGrids and polygons saved.")

C:\Users\Aska\AppData\Local\Temp\ipykernel_30040\1812961312.py:27: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  slope_arr = slope_src.read(1, window=win).astype(float)
C:\Users\Aska\AppData\Local\Temp\ipykernel_30040\1812961312.py:28: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  elev_arr  = elev_src.read(1,  window=win).astype(float)


Nobile Rim 2               grid shape: (602, 602)  transform origin: (145160, 107200)
Mons Mouton                grid shape: (502, 502)  transform origin: (63040, 128240)
Malapert Massif            grid shape: (626, 626)  transform origin: (-13040, 134480)
de Gerlache Rim 2          grid shape: (600, 600)  transform origin: (-61400, 35360)
Mons Mouton Plateau        grid shape: (2018, 2018)  transform origin: (47440, 192560)
Slater Plain               grid shape: (600, 600)  transform origin: (58440, -38320)
Peak near Cabeus B         grid shape: (600, 600)  transform origin: (-192280, 82120)
Nobile Rim 1               grid shape: (600, 600)  transform origin: (72200, 122360)
Haworth                    grid shape: (846, 846)  transform origin: (-55320, 108520)

Grids and polygons saved.
